In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task
Load the dataset "Metro_Interstate_Traffic_Volume.csv" from "/content/drive/MyDrive/Metro_Interstate_Traffic_Volume.csv" into a pandas DataFrame, ensuring it is properly parsed with the correct separator (`,`) and decimal point (`.`), and `iso-8859-1` encoding. Display the head of the DataFrame for initial inspection.

## Set Dataset Path and Load Data

### Subtask:
Explicitly set the `DATASET_PATH` to `/content/drive/MyDrive/Metro_Interstate_Traffic_Volume.csv`. Load the dataset from this path into a pandas DataFrame, ensuring correct separator and decimal point handling (`;` and `,` respectively) and proper encoding ('iso-8859-1'). Display the head of the DataFrame for initial inspection.


**Reasoning**:
The subtask requires setting a specific dataset path and then loading the data into a DataFrame with specific parsing parameters. I will create a code block to explicitly set the `DATASET_PATH` and then use `pd.read_csv` with the specified `sep`, `decimal`, and `encoding` to load the `df`, finally displaying its head.



In [52]:
DATASET_PATH = "/content/drive/MyDrive/Metro_Interstate_Traffic_Volume.csv"
df = pd.read_csv(DATASET_PATH, sep=',', decimal='.', encoding='iso-8859-1')
display(df.head())

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


## Initial Data Preprocessing and NaN Handling

### Subtask:
Convert the `date_time` column to datetime objects. Check for any missing values across all columns and handle them appropriately, potentially using forward-fill or interpolation if necessary, as the previous -200.0 placeholders are not expected in this dataset. Display the info and missing value counts to confirm data readiness.


**Reasoning**:
The subtask requires converting the `date_time` column to datetime objects, displaying DataFrame information, and showing the count of missing values for each column. This code block will perform these three steps sequentially.



In [53]:
df['date_time'] = pd.to_datetime(df['date_time'])
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              61 non-null     object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 3.3+ MB

Missing values per column:
holiday                48143
temp                       0
rain_1h                    0
snow_1h                    0
clouds_all                 0
weather_mai

**Reasoning**:
The 'holiday' column has a significant number of missing values (48143 out of 48204). Given the nature of this categorical feature, replacing these missing values with 'None' or 'No Holiday' is a more appropriate strategy than imputation methods like forward-fill or interpolation, which would not make sense for a categorical column with such sparse data. The other columns do not have missing values, so no further action is needed for them.



In [54]:
df['holiday'] = df['holiday'].fillna('None')
print("Missing values after handling holiday column:")
print(df.isnull().sum())

Missing values after handling holiday column:
holiday                0
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
date_time              0
traffic_volume         0
dtype: int64


## Feature Engineering

### Subtask:
Extract 'hour' and 'day_of_week' from the `date_time` column and add them as new numerical features. Additionally, handle categorical features such as `holiday`, `weather_main`, and `weather_description` using one-hot encoding to convert them into a numerical format suitable for the model.


**Reasoning**:
The subtask requires extracting 'hour' and 'day_of_week' from the 'date_time' column and then applying one-hot encoding to the 'holiday', 'weather_main', and 'weather_description' categorical columns. Finally, the head and info of the modified DataFrame will be displayed to verify the changes.



In [55]:
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek

categorical_cols = ['holiday', 'weather_main', 'weather_description']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("DataFrame head after feature engineering:")
display(df.head())

print("\nDataFrame info after feature engineering:")
df.info()

DataFrame head after feature engineering:


,temp,rain_1h,snow_1h,clouds_all,date_time,traffic_volume,hour,day_of_week,holiday_Columbus Day,holiday_Independence Day,...,weather_description_sleet,weather_description_smoke,weather_description_snow,weather_description_thunderstorm,weather_description_thunderstorm with drizzle,weather_description_thunderstorm with heavy rain,weather_description_thunderstorm with light drizzle,weather_description_thunderstorm with light rain,weather_description_thunderstorm with rain,weather_description_very heavy rain
0,288.28,0.0,0.0,40,2012-10-02 09:00:00,5545,9,1,False,False,...,False,False,False,False,False,False,False,False,False,False
1,289.36,0.0,0.0,75,2012-10-02 10:00:00,4516,10,1,False,False,...,False,False,False,False,False,False,False,False,False,False
2,289.58,0.0,0.0,90,2012-10-02 11:00:00,4767,11,1,False,False,...,False,False,False,False,False,False,False,False,False,False
3,290.13,0.0,0.0,90,2012-10-02 12:00:00,5026,12,1,False,False,...,False,False,False,False,False,False,False,False,False,False
4,291.14,0.0,0.0,75,2012-10-02 13:00:00,4918,13,1,False,False,...,False,False,False,False,False,False,False,False,False,False



DataFrame info after feature engineering:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 66 columns):
 #   Column                                                   Non-Null Count  Dtype         
---  ------                                                   --------------  -----         
 0   temp                                                     48204 non-null  float64       
 1   rain_1h                                                  48204 non-null  float64       
 2   snow_1h                                                  48204 non-null  float64       
 3   clouds_all                                               48204 non-null  int64         
 4   date_time                                                48204 non-null  datetime64[ns]
 5   traffic_volume                                           48204 non-null  int64         
 6   hour                                                     48204 non-null  int32         
 7   day_of

## Scale Numerical Features

### Subtask:
Identify all numerical columns in the processed DataFrame (including the newly engineered and one-hot encoded features, and the target column). Instantiate a StandardScaler and apply it to scale these features. Store the StandardScaler instance, as it will be needed later for inverse transformation of the target column.


**Reasoning**:
First, I'll drop the 'date_time' column as it has served its purpose in feature engineering. Then, I'll identify all numerical feature columns, excluding 'traffic_volume' and scale them using `StandardScaler`. After that, I'll scale the 'traffic_volume' column separately with its own `StandardScaler` instance and store it for future inverse transformations. Finally, I'll display the head and information of the DataFrame to verify the scaling.



In [56]:
df = df.drop('date_time', axis=1)

# Identify numerical columns for features (excluding 'traffic_volume')
feature_cols = df.columns.drop('traffic_volume')

# Instantiate and apply StandardScaler for features
feature_scaler = StandardScaler()
df[feature_cols] = feature_scaler.fit_transform(df[feature_cols])

# Instantiate and apply StandardScaler for 'traffic_volume'
traffic_scaler = StandardScaler()
df['traffic_volume'] = traffic_scaler.fit_transform(df[['traffic_volume']])

print("DataFrame head after scaling:")
display(df.head())

print("\nDataFrame info after scaling:")
df.info()

DataFrame head after scaling:


,temp,rain_1h,snow_1h,clouds_all,traffic_volume,hour,day_of_week,holiday_Columbus Day,holiday_Independence Day,holiday_Labor Day,...,weather_description_sleet,weather_description_smoke,weather_description_snow,weather_description_thunderstorm,weather_description_thunderstorm with drizzle,weather_description_thunderstorm with heavy rain,weather_description_thunderstorm with light drizzle,weather_description_thunderstorm with light rain,weather_description_thunderstorm with rain,weather_description_very heavy rain
0,0.530370,-0.007463,-0.027228,-0.239963,1.150159,-0.345548,-0.989561,-0.010185,-0.010185,-0.012051,...,-0.007889,-0.020373,-0.078202,-0.050989,-0.006441,-0.036175,-0.017643,-0.033489,-0.027716,-0.019328
1,0.611341,-0.007463,-0.027228,0.657120,0.632251,-0.201459,-0.989561,-0.010185,-0.010185,-0.012051,...,-0.007889,-0.020373,-0.078202,-0.050989,-0.006441,-0.036175,-0.017643,-0.033489,-0.027716,-0.019328
2,0.627836,-0.007463,-0.027228,1.041584,0.758582,-0.057371,-0.989561,-0.010185,-0.010185,-0.012051,...,-0.007889,-0.020373,-0.078202,-0.050989,-0.006441,-0.036175,-0.017643,-0.033489,-0.027716,-0.019328
3,0.669071,-0.007463,-0.027228,1.041584,0.888940,0.086718,-0.989561,-0.010185,-0.010185,-0.012051,...,-0.007889,-0.020373,-0.078202,-0.050989,-0.006441,-0.036175,-0.017643,-0.033489,-0.027716,-0.019328
4,0.744794,-0.007463,-0.027228,0.657120,0.834582,0.230807,-0.989561,-0.010185,-0.010185,-0.012051,...,-0.007889,-0.020373,-0.078202,-0.050989,-0.006441,-0.036175,-0.017643,-0.033489,-0.027716,-0.019328



DataFrame info after scaling:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 65 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   temp                                                     48204 non-null  float64
 1   rain_1h                                                  48204 non-null  float64
 2   snow_1h                                                  48204 non-null  float64
 3   clouds_all                                               48204 non-null  float64
 4   traffic_volume                                           48204 non-null  float64
 5   hour                                                     48204 non-null  float64
 6   day_of_week                                              48204 non-null  float64
 7   holiday_Columbus Day                                     48204 non-null  float64


## Define create_sequences Function

### Subtask:
Define the `create_sequences` function, which is a generic utility for converting time series data into a supervised learning format. This function will accept the DataFrame, target column name, look-back window, forecast horizon, and a list of features, then return `X` (input sequences) and `y` (target sequences).


**Reasoning**:
The subtask requires defining the `create_sequences` function, which converts time series data into a supervised learning format. This function will extract input and target sequences based on the look-back window and forecast horizon from the provided DataFrame.



In [57]:
def create_sequences(data, target_column, look_back_window, forecast_horizon, features):
    X, y = [], []
    y_data = data[target_column].values

    for i in range(len(data) - look_back_window - forecast_horizon + 1):
        # Extract input sequence X_seq from features columns
        X_seq = data[features].iloc[i:(i + look_back_window)].values

        # Extract target sequence y_seq from the target column
        y_seq = y_data[(i + look_back_window):(i + look_back_window + forecast_horizon)]

        X.append(X_seq)
        y.append(y_seq)

    return np.array(X), np.array(y)

print("The 'create_sequences' function has been defined.")

The 'create_sequences' function has been defined.


## Define Model Parameters and Create Sequences

### Subtask:
Set the `TARGET_COLUMN` to `traffic_volume`. Define `look_back_window` (e.g., 24 for daily patterns) and `forecast_horizon` (e.g., 1 for single-step prediction). Use the `create_sequences` function with the scaled DataFrame and these parameters to generate the `X` (input) and `y` (target) sequences for the model.


**Reasoning**:
The subtask requires defining model parameters and then using the `create_sequences` function to generate input and target sequences. I will define the target column, look-back window, and forecast horizon, then prepare the features list, call the function, and finally print the shapes of the resulting arrays.



In [58]:
TARGET_COLUMN = 'traffic_volume'
look_back_window = 24
forecast_horizon = 1

FEATURES = [col for col in df.columns if col != TARGET_COLUMN]

X, y = create_sequences(df, TARGET_COLUMN, look_back_window, forecast_horizon, FEATURES)

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (48180, 24, 64)
Shape of y: (48180, 1)


## Split Data into Training and Test Sets

### Subtask:
Divide the generated X and y sequences into training and testing sets. Implement a chronological split, typically using 80% of the data for training and the remaining 20% for testing, to preserve the temporal dependencies inherent in time series data.


**Reasoning**:
The subtask requires splitting the `X` and `y` sequences into training and testing sets chronologically using an 80/20 ratio. I will calculate the split index, perform the array splits, and then print the shapes of the resulting training and testing sets to verify the operation.



In [59]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print('Shape of X_train:', X_train.shape)
print('Shape of y_train:', y_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)

Shape of X_train: (38544, 24, 64)
Shape of y_train: (38544, 1)
Shape of X_test: (9636, 24, 64)
Shape of y_test: (9636, 1)


## Build and Compile LSTM Model

### Subtask:
Construct the LSTM neural network model. The architecture should include an Input layer, an LSTM layer (e.g., with 50 units and 'relu' activation), and a Dense output layer, with the output units matching the forecast_horizon. Compile the model using the 'adam' optimizer and 'mean_squared_error' as the loss function, while monitoring 'mean_absolute_error' during training.


**Reasoning**:
The subtask requires building and compiling an LSTM model with specific architecture and compilation parameters. I will define a Sequential model, add an LSTM layer with the specified units, activation, and input shape, then add a Dense output layer. Finally, I will compile the model using the Adam optimizer, mean_squared_error loss, and mean_absolute_error metric, and display the model summary.



In [60]:
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(forecast_horizon))

optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mean_absolute_error'])

print("LSTM Model Summary:")
model.summary()

LSTM Model Summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 50)             │        23,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,051 (90.04 KB)

 Trainable params: 23,051 (90.04 KB)

 Non-trainable params: 0 (0.00 B)

In [63]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)
print('Model training complete.')

Epoch 1/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 25s 19ms/step - loss: 0.5865 - mean_absolute_error: 0.5742 - val_loss: 0.2133 - val_mean_absolute_error: 0.3412
Epoch 2/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.2258 - mean_absolute_error: 0.3490 - val_loss: 0.1826 - val_mean_absolute_error: 0.2902
Epoch 3/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 17s 14ms/step - loss: 0.1701 - mean_absolute_error: 0.2924 - val_loss: 0.1399 - val_mean_absolute_error: 0.2602
Epoch 4/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.1390 - mean_absolute_error: 0.2620 - val_loss: 0.1214 - val_mean_absolute_error: 0.2418
Epoch 5/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - loss: 0.1269 - mean_absolute_error: 0.2476 - val_loss: 0.1078 - val_mean_absolute_error: 0.2265
Epoch 6/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - loss: 0.1146 - mean_absolute_error: 0.2316 - val_loss: 0.1140 - val_mean_absolute_error: 0.2316
Epoch 7/50
1205/1205 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.1030 - mea